# Question Answering System with RetrievalQAWithSourcesChain

An end-to-end question answering pipeline over real-world news articles using **LangChain**, **Google Generative AI (Gemini)**, and **FAISS** vector store.

---

## Setup & API Key Configuration

In [1]:
import os
import streamlit as st
import pickle
import time
import langchain
from langchain import OpenAI
from langchain.chains import RetrievalQAWithSourcesChain
from langchain.chains.qa_with_sources.loading import load_qa_with_sources_chain
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import UnstructuredURLLoader
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

C:\Users\msi1\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
# Load Gemini API key
import os
from secret_key import GOOGLE_API_KEY

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [4]:
import os
import streamlit as st
import pickle
import time
import langchain

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain.chains import RetrievalQAWithSourcesChain
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import UnstructuredURLLoader
from langchain.vectorstores import FAISS

In [5]:
# Load Gemini API key
from secret_key import GOOGLE_API_KEY

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

## (1) Load data

Load article text directly from financial news URLs using `UnstructuredURLLoader`.

In [10]:
loaders = UnstructuredURLLoader(urls=[
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
])
data = loaders.load() 
len(data)

2

## (2) Split data to create chunks

Split documents into smaller chunks using `RecursiveCharacterTextSplitter` so each chunk fits within token limits while preserving context.

In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# As data is of type documents we can directly use split_documents over split_text in order to get the chunks.
docs = text_splitter.split_documents(data)

In [13]:
len(docs)

19

In [14]:
docs[0]

Document(metadata={'source': 'https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html'}, page_content='English\n\nHindi\n\nGujarati\n\nSpecials\n\nLoan Offers\n\nMy Alerts\n\nGo Ad-Free\n\nHello, Login\n\nHello, Login\n\nLog-inor Sign-Up\n\nMy Account\n\nMy Profile\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nMy Profile\n\nMy PRO\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nLogout\n\nLoans up to ₹50 LAKHS\n\nFixed Deposits\n\nCredit CardsLifetime Free\n\nCredit Score\n\nLoan against MFs\n\nChat with Us\n\nDownload App\n\nFollow us on:\n\nNetwork 18\n\n>->MC_ENG_DESKTOP/MC_ENG_NEWS/MC_ENG_MARKETS_AS/MC_ENG_ROS_NWS_MKTS_AS_ATF_728\n\nMoneycontrol\n\nGo PRO NowPRO\n\nMoneycontrol AD Lite\n\nMoneycontrol AD Lite\n\nMoneycontrol PRO\n\nMoneycontrol Super PRO\n\nAdvertisement\n\nRemove Ad\n\nBusiness\n\nMarkets\n\nStocks\n\nEconomy\n\nCompanies\n\nTrends\n\nIPO\n\nOpin

## (3) Create embeddings for these chunks and save them to FAISS index

Generate vector embeddings for each chunk and persist them to a local FAISS vector store.

In [17]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    google_api_key=GOOGLE_API_KEY
)

vectorindex_gemini = FAISS.from_documents(docs, embeddings)

In [20]:
# Store FAISS vector index locally
vectorindex_gemini.save_local("vector_index")

# Load FAISS vector index later
vectorindex_gemini = FAISS.load_local(
    "vector_index",
    embeddings,
    allow_dangerous_deserialization=True
)

## (4) Retrieve similar embeddings for a given question and call LLM to retrieve final answer

Initialize the LLM and set up `RetrievalQAWithSourcesChain` to retrieve similar chunks from the FAISS vector index, generate a contextual response, and cite the source URLs.

In [24]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)

In [25]:
chain = RetrievalQAWithSourcesChain.from_llm(
    llm=llm,
    retriever=vectorindex_gemini.as_retriever()
)

chain

RetrievalQAWithSourcesChain(combine_documents_chain=MapReduceDocumentsChain(llm_chain=LLMChain(prompt=PromptTemplate(input_variables=['context', 'question'], template='Use the following portion of a long document to see if any of the text is relevant to answer the question. \nReturn any relevant text verbatim.\n{context}\nQuestion: {question}\nRelevant text, if any:'), llm=ChatGoogleGenerativeAI(model='models/gemini-3.5-flash-lite', google_api_key=SecretStr('**********'), temperature=0.0, client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x000002AFB4BB2290>, async_client=<google.ai.generativelanguage_v1beta.services.generative_service.async_client.GenerativeServiceAsyncClient object at 0x000002AFB4BB01C0>, default_metadata=())), reduce_documents_chain=ReduceDocumentsChain(combine_documents_chain=StuffDocumentsChain(llm_chain=LLMChain(prompt=PromptTemplate(input_variables=['question', 'summaries'], template='Given the follow

### Querying the System & Inspecting Results

In [26]:
query = "what is the price of Tiago iCNG?"
# query = "what are the main features of punch iCNG?"

langchain.debug=True

chain({"question": query}, return_only_outputs=True)

C:\Users\msi1\AppData\Local\Temp\ipykernel_8612\377114326.py:6: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  chain({"question": query}, return_only_outputs=True)


[chain/start] [chain:RetrievalQAWithSourcesChain] Entering Chain run with input:
{
  "question": "what is the price of Tiago iCNG?"
}


Error in ConsoleCallbackHandler.on_chain_start callback: ValidationError(model='Run', errors=[{'loc': ('__root__',), 'msg': "argument of type 'NoneType' is not iterable", 'type': 'type_error'}])
Parent run be2a1eb1-60b5-4220-9504-2acf55ea9901 not found for run 7ab88adc-d04f-4504-a9aa-68507f3b310e. Treating as a root run.
Parent run be2a1eb1-60b5-4220-9504-2acf55ea9901 not found for run d58f0e89-7cfe-4f7d-8207-b69304c68d07. Treating as a root run.
Parent run be2a1eb1-60b5-4220-9504-2acf55ea9901 not found for run bb301753-db95-4f81-b5c8-a895bf279369. Treating as a root run.
Parent run be2a1eb1-60b5-4220-9504-2acf55ea9901 not found for run 7375ca21-6f56-4bcc-a7a7-e2368ee99824. Treating as a root run.


[chain/start] [chain:RetrievalQAWithSourcesChain > chain:MapReduceDocumentsChain] Entering Chain run with input:
[inputs]
[llm/start] [llm:ChatGoogleGenerativeAI] Entering LLM run with input:
{
  "prompts": [
    "Human: Use the following portion of a long document to see if any of the text is relevant to answer the question. \nReturn any relevant text verbatim.\nTata Motors launches Punch iCNG, price starts at Rs 7.1 lakh\n\nTEL\n\nWatchlist\n\nPortfolio\n\nMessage\n\nFollow\n\nDay High\n\nDay Low\n\nVolume (NSE)\n\nMore\n\nAdd to Watchlist\n\nTata Motors on Friday launched the CNG variant of its micro SUV Punch priced between Rs 7.1 lakh and Rs 9.68 lakh (ex-showroom, Delhi).\n\nThe Punch iCNG is equipped with the company's proprietary twin-cylinder technology with enhanced safety features like a micro-switch to keep the car switched off at the time of refuelling and thermal incident protection that cuts off CNG supply to the engine and releases gas into the atmosphere, Tata Motors s

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised DeadlineExceeded: 504 Deadline Exceeded.
Error in ConsoleCallbackHandler.on_chain_end callback: TracerException('No indexed run ID be2a1eb1-60b5-4220-9504-2acf55ea9901.')


[llm/end] [llm:ChatGoogleGenerativeAI] [625.84s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "Relevant text, if any:\nNone",
        "generation_info": {
          "finish_reason": "STOP",
          "safety_ratings": []
        },
        "type": "ChatGeneration",
        "message": {
          "lc": 1,
          "type": "constructor",
          "id": [
            "langchain",
            "schema",
            "messages",
            "AIMessage"
          ],
          "kwargs": {
            "content": "Relevant text, if any:\nNone",
            "response_metadata": {
              "prompt_feedback": {
                "block_reason": 0,
                "safety_ratings": []
              },
              "finish_reason": "STOP",
              "safety_ratings": []
            },
            "type": "ai",
            "id": "run-7ab88adc-d04f-4504-a9aa-68507f3b310e-0",
            "usage_metadata": {
              "input_tokens": 264,
              "out

{'answer': "I don't know the answer.\n\n",
 'sources': 'https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html'}

The retrieval chain successfully retrieves the relevant context from the FAISS vector index and generates the final answer along with reference `SOURCES`.